In [214]:
import yaml
import copy
from datetime import datetime,timezone,timedelta


In [215]:
def load_yaml(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

In [216]:
config          = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml") 

In [217]:
op1 = {
    'section': 'Profile',
    'scores': {
        'ContentQuality': {
            'score': 1,
            'feedback': "xxx"},
        'Completeness': {
            'score': 0,
            'feedback': "yyy"}
        },
    'session_feedback': 'zzz'
}

In [218]:
op3 = {
    'section': 'Education',
    'scores': {
        'RoleRelevance': {
            'score': 5,
            'feedback': 'abc'},
        'Completeness': {
        'score': 5,
        'feedback': "def"}
        },
    'session_feedback': 'ghi'
}

In [219]:
# Now score from llm is 1-5 but when It's return 0 It's mean there are no
# that section so do calculate it 

In [232]:
def aggregate(llm_output:dict,verbose=0):
    '''
    convert, reshape, transform output format that We got from llm
    '''
    # print(f"llm_output ->\n{llm_output}")
    llm_output       = llm_output                  # op
    section_name     = llm_output["section"]       # Get section
    section_scores   = config["weights"][section_name]  # config["weights"][section_key][criteria]
    ddict = {} 
    total_score = 0.0
    full_score  = 0.0
    scores_copy = copy.deepcopy(llm_output["scores"])  # Protect multiple mutation when we run more than one time
    for criteria,body in scores_copy.items():
        raw_score_from_llm = body["score"]             # score ดิบๆ ที่ออกมาจาก LLM
        if raw_score_from_llm == 0:
            full_score_from_config = 0
        else:
            full_score_from_config = section_scores[criteria]   # weight ที่ set ใน weight.yaml เพื่อ scale เต็มๆ
        scored = raw_score_from_llm / 5 * full_score_from_config          # raw / normalize max score * weight scale
        body["score"] = scored
        ddict[criteria] = body
        total_score = total_score + scored
        full_score  = full_score  + full_score_from_config
        if verbose:
            print(f"criteria->{criteria}")
            print(f"body->{body}")
            print(f" - raw -> {raw}") 
            print(f" - w   -> {w}")
            print(f" - weighted -> {weighted}")  
            print(f" - body['score'] -> {body['score']}")  
            print(f"total -> {total}")
            print()
    return {
            "section": section_name,
            "total_score":total_score,
            "full_score":full_score,
            "scores":ddict,
            "session_feedback":llm_output['session_feedback']
        }

In [233]:
print("s1")
s1 = aggregate(op1)
print("s2")
s2 = aggregate(op3)

s1
s2


In [234]:
s1

{'section': 'Profile',
 'total_score': 2.0,
 'full_score': 10.0,
 'scores': {'ContentQuality': {'score': 2.0, 'feedback': 'xxx'},
  'Completeness': {'score': 0.0, 'feedback': 'yyy'}},
 'session_feedback': 'zzz'}

In [223]:
s2

{'section': 'Education',
 'total_score': 20.0,
 'full_score': 20.0,
 'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'},
  'Completeness': {'score': 10.0, 'feedback': 'def'}},
 'session_feedback': 'ghi'}

<hr>

In [235]:
section_outputs = [s1,s2]
timestamp       = str(datetime.now(tz=(timezone(timedelta(hours=7)))))
model_config    = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\model.yaml")     # should include model name
weight_config   = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml")    # includes weights + version
prompt_config   = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompt.yaml") 
config_lang     = prompt_config['Language_output_style']["en"]

In [236]:
section_outputs

[{'section': 'Profile',
  'total_score': 2.0,
  'full_score': 10.0,
  'scores': {'ContentQuality': {'score': 2.0, 'feedback': 'xxx'},
   'Completeness': {'score': 0.0, 'feedback': 'yyy'}},
  'session_feedback': 'zzz'},
 {'section': 'Education',
  'total_score': 20.0,
  'full_score': 20.0,
  'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'},
   'Completeness': {'score': 10.0, 'feedback': 'def'}},
  'session_feedback': 'ghi'}]

In [226]:
every_section_score_total = 0
every_section_full_score_total = 0

for section_data in section_outputs:
    section_name                   = section_data["section"]
    section_total_score            = section_data["total_score"]
    section_full_score             = section_data["full_score"]
    every_section_score_total      = every_section_score_total + section_total_score
    every_section_full_score_total = every_section_full_score_total + section_full_score

In [227]:
every_section_score_total

22.0

In [228]:
every_section_full_score_total

30.0

<hr>

In [229]:
# 90–100 /100 -> A
# 80–89  /100 -> B
# 70–79  /100 -> C
# 60–69  /100 -> D
# 00–49  /100 -> F

def normalize_score(point_score,full_score):
    print(f"Point_score -> {point_score}")
    print(f"full_score  -> {full_score}")
    normalize_score = (point_score/full_score) * 100
    print(f"normalize_score -> {normalize_score}/100")
    return normalize_score

def score_to_grade(score: float) -> str:
    if 90 <= score <= 100:
        return "A"
    elif 80 <= score < 90:
        return "B"
    elif 70 <= score < 80:
        return "C"
    elif 60 <= score < 70:
        return "D"
    elif 0 <= score < 60:
        return "F"
    else:
        return "Gradding error"

# # 40/40*100 -> 100.0
# normalized_score = normalize_score(40,40)
# score_to_grade(normalized_score)

# # 40/40*100 -> 100.0
# normalized_score = normalize_score(50,100)
# score_to_grade(normalized_score)

# # 30/40 * 100
# normalize_score(30,40)
# score_to_grade(normalized_score)

In [230]:
# 40/40*100 -> 100.0
normalized_score = normalize_score(every_section_score_total,every_section_full_score_total)
score_to_grade(normalized_score)

Point_score -> 22.0
full_score  -> 30.0
normalize_score -> 73.33333333333333/100


'C'

Report 00
- คอมมีปัญหาหน้าจอดับ

Report 01
- Ticket ของ week ที่แล้วเสร็จหมดแล้วตอนนี้อยู่ Branch dev
- ตอนนี้ทุกอย่างเป็น Promptbuilder -> PromptBuilder + Session,Global feedback + PromptSplit = BasePromptBuilder
- ยิง 300 - 400 ครั้ง error 1 ครับ
- PowerBI report
- ทดสอบทั้งหมด 3 branch branch ละ 100 example = 300 examples
- ตอนนี้ไม่ได้ Deploy ขึ้นเพราะอยากสร้าง Endpoint ใหม่

In [ ]:
Report 02
- Rethink?

Question 01
- input ของแต่ละ CV,Resume เนี้ยอยู่ในระบบ Web ใช่ไหม ?
     - การ Detect และตรวจสอบว่า User ขาดอะไรให้ทำเป็นแบบ JSON + Pure python bc เร็วกว่า LLM ไม่งั้นจะเป็นการเพิ่ม LLM ไปอีก 1 ชั้น
- ถ้าอย่างนั้นมันต้องมีการกำหนด Template กลางระหว่าง Dev & Data
- อันนี้เป็นตัวอย่างของ CV Resume
- ถ้านับรวมตรงมีโอกาสไม่สมบูรณ์เพราะต้องไป align with dev team

Question 02 : Adaptive score
- Figma process workflow
- มีการรื้อหลายจุด และ Modify
    - ?มีการเพิ่ม คลาสใหม่
    - aggregator.py เป็นหลัก โดย สิ่งที่ออกมาจาก LLM ->
        - 0 : ไม่มี
        - 1-5 : grade scale
      ตอนนี้คือสร้างสิ่งนึ่งที่คอยเก็บ score ที่เป็นไปได้ทั้งหมดว่า
    - การทำให้มัน adaptive เนี้ยจะเอาระดับไหน
        - Section  ( )
        - Criteria (x)
        - Item     ( )
    - ถ้าระดับ Criteria 
    - ถ้าระดับ Item

Question 03 : Output
- ตอนนี้สิ่งที่ออกมาจาก LLM คือ


In [ ]:
{
    "Profile": {
        "name": "Juan Jose Carin",
        "email": "juanjose.carin@gmail.com",
        "phone": "650-336-4590",
        "linkedin": "linkedin.com/in/juanjosecarin",
        "jobdb_link": "-",
        "portfolio_link": "juanjocarin.github.io"
    },
    "Summary": {
        "has_summary": "Yes",
        "summary_points": [
            "Passionate about data analysis and experiments, mainly focused on user behavior, experience, and engagement.",
            "Solid background in data science and statistics, with extensive experience using data insights to drive business growth."
        ]
    },
    "Education": [
        {
            "institution": "University of California, Berkeley",
            "degree": "Master of Information and Data Science",
            "dates": "2016",
            "gpa": "3.93",
            "honors": "Relevant courses: Machine Learning; Machine Learning at Scale; Storing and Retrieving Data; Field Experiments; Applied Regression and Time Series Analysis; Exploring and Analyzing Data; Data Visualization and Communication; Research Design and Applications for Data Analysis."
        },
        {
            "institution": "Universidad Politécnica de Madrid",
            "degree": "M.S. in Statistical and Computational Information Processing",
            "dates": "2014",
            "gpa": "3.69",
            "honors": "Relevant courses: Data Mining; Multivariate Analysis; Time Series; Neural Networks and Statistical Learning; Regression and Prediction Methods; Optimization Techniques; Monte Carlo Techniques; Numerical Methods in Finance; Stochastic Models in Finance; Bayesian Networks."
        },
        {
            "institution": "Universidad Politécnica de Madrid",
            "degree": "M.S. in Telecommunication Engineering",
            "dates": "2005",
            "gpa": "3.03",
            "honors": "Focus Area: Radio communication systems (radar and mobile). Fellowship: First year at University, due to Honors obtained last year at high school."
        }
    ],
    "Experience": [
        {
            "title": "Data Scientist",
            "company": "CONENTO (Madrid, Spain) — working remotely",
            "dates": "Jan 2016 - Mar 2016",
            "description": [
                "Designed and implemented the ETL pipeline for a predictive model of traffic on the main roads in eastern Spain (project for the Spanish government).","Automated scripts in R to extract, transform, clean (including anomaly detection), and load into MySQL data from multiple sources (road traffic sensors, accidents, road works, weather)."
            ]
        },
        {
            "title": "Data Scientist",
            "company": "CONENTO (Madrid, Spain)",
            "dates": "Jun 2014 - Sep 2014",
            "description": [
                "Designed an experiment for Google Spain (conducted in Oct 2014) to measure the impact of YouTube ads on the sales of a car manufacturer's dealer network.","Used a matched-pair, cluster-randomized design selecting test/control groups from 50+ cities based on sales-wise similarity over time (using wavelets and R)."
            ]
        },
        {
            "title": "Head of Sales, Spain & Portugal - Test & Measurement dept.",
            "company": "YOKOGAWA (Madrid, Spain)",
            "dates": "Feb 2009 - Aug 2013",
            "description": [
                "Applied analysis of sales and market trends to decide the direction of the department.","Led a team of 7 people.","Increased revenue by 6.3%, gross profit by 4.2%, and operating income by 146%; achieved a 30% ratio of new customers (3x growth) by entering new markets and improving customer service and training."
            ]
        },
        {
            "title": "Sales Engineer - Test & Measurement dept.",
            "company": "YOKOGAWA (Madrid, Spain)",
            "dates": "Apr 2008 - Jan 2009",
            "description": [
                "Promoted to head of sales after 5 months leading the sales team."
            ]
        },
        {
            "title": "Sales & Application Engineer",
            "company": "AYSCOM (Madrid, Spain)",
            "dates": "Sep 2004 - Mar 2008",
            "description": ["Exceeded sales target every year from 2005 to 2007; achieved 60% of the target in the first 3 months of 2008."]
        },
        {
            "title": "Tutor of Differential & Integral Calculus, Physics, and Digital Electronic Circuits",
            "company": "ACADEMIA UNIVERSITARIA (Madrid, Spain)",
            "dates": "Jul 2002 - Jun 2004",
            "description": [
                "Highest-rated professor in student surveys in 4 of the 6 terms.",
                "Increased ratio of students passing the course by 25%."
            ]
        }
    ],
    "Activities": [
        {
            "name": "LLM Evaluation Harness (Internal Tooling)",
            "details": "Built a golden-dataset based evaluation harness (rubric scoring, regression checks, release gates) for RAG and resume-evaluation prompts; improved release confidence and reduced regressions by ~60%.",
            "tech_stack": ["Python", "pytest", "BigQuery", "RAG"]
        },
        {
            "name": "Knowledge Sharing / Tech Talks",
            "details": "Ran internal workshops on orchestrator architecture, API reliability patterns, and production LLM practices (prompt versioning, caching, monitoring)."
        },
        {
            "name": "Open-source / Internal Libraries",
            "details": "Contributed reusable utilities for schema validation and JSON naming conversion; reduced integration bugs and improved developer onboarding."
            }
        ],
    "Skills": {
        "technical": [
            "Data Analysis","Statistics","Experiment Design","ETL","Anomaly Detection","MySQL","Machine Learning","Data Visualization","Big Data"
        ],
        "soft_skills": [
            "Team Leadership","Customer Service Improvement","Training & Mentoring"
        ],
        "tools_with_levels": [
            {"tool": "Python","level": "proficient"},
            {"tool": "R","level": "proficient"},
            {"tool": "SQL","level": "proficient"},
            {"tool": "Hadoop","level": "proficient"},
            {"tool": "Hive","level": "proficient"},
            {"tool": "MrJob","level": "proficient"},
            {"tool": "Tableau","level": "proficient"},
            {"tool": "Git","level": "proficient"},
            {"tool": "AWS","level": "proficient"},
            {"tool": "Spark","level": "intermediate"},
            {"tool": "Storm","level": "intermediate"},
            {"tool": "Bash","level": "intermediate"},
            {"tool": "SPSS","level": "intermediate"},
            {"tool": "SAS","level": "intermediate"},
            {"tool": "Matlab","level": "intermediate"},
            {"tool": "EViews","level": "basic"},
            {"tool": "Demetra+","level": "basic"},
            {"tool": "D3.js","level": "basic"},
            {"tool": "Gephi","level": "basic"},
            {"tool": "Neo4j","level": "basic"},
            {"tool": "QGIS","level": "basic"}
        ],
        "languages": [
            {"language": "Python","level": "proficient"},
            {"language": "R","level": "proficient"},
            {"language": "SQL","level": "proficient"},
            {"language": "Bash","level": "intermediate"},
            {"language": "Matlab","level": "intermediate"}
        ]
    }
}